# 09b — Verifier: Full 7-Step Normalization (Move 5)

**Phase 9b** — runs after `09_search_subgraph.ipynb`.

## What this notebook does

1. **Pre-flight** — verifies Neo4j connectivity, confirms verifier + normalizer are importable, checks CHUNK count and `textCanonical` coverage.
2. **7-step normalization walkthrough** — demonstrates each step (NFC → whitespace → T-S → 異體字 → 避諱 → 通假字 → mojimoji) on sample classical-Chinese text, showing per-step diffs.
3. **Move 5 contract — three test cases** (plan A3):
   - **Test A**: faithful span (exact substring of a real chunk's text) → **passes**.
   - **Test B**: T-S simplified variant span (only matches *after* `s2t` normalization) → **passes after normalization**.
   - **Test C**: fabricated span (text that is not in the chunk) → **fails → `INSUFFICIENT_EVIDENCE` + `VERIFIER_FAILURE` node written**.
4. **Live corpus verification** — samples 10 real chunks and runs each against its own text (all should pass); shows failure rate.
5. **Evidence-strength badges** — shows `primary_source`, `primary_疏議`, `editorial_commentary`, `scholarly_interpretation` assignment logic.
6. **VERIFIER_FAILURE inspection** — queries Neo4j for `(:VERIFIER_FAILURE)` nodes written in this session.
7. **Search pipeline integration** — runs a demo query via `search()`, pipes top result through `verify_cite()`.
8. **Artefact write** — saves `notebooks/_artifacts/09b_verifier/verifier.json`.

## Production modules

| Module | Role |
|---|---|
| `apps.backend.agents.verifier` | Deterministic cite gate (plan A3, Move 5) |
| `apps.backend.normalize.pipeline` | 7-step normalization (plan §2.7) |
| `apps.backend.normalize.tsc` | T-S step (OpenCC `s2t` / `t2s`) |
| `apps.backend.agents.relevance_scorer` | Old LLM scorer (renamed, still used for soft ranking) |

## Note on `textCanonical` coverage

`logs/corpus_state.json` shows **0% `textCanonical` coverage** — the bulk translation run (Track A, A1–A2) has not yet completed.  The verifier falls back to `c.text` (raw OCR / native text) when `textCanonical` is null; all test cases in §3 run against raw text.  Once `scripts/run_translation.py` completes, re-running §4–7 will exercise the full canonical path.

## Move 5 guarantee

> Every cite returned by the search pipeline either:
> (a) resolves to a CHUNK whose normalized text contains the normalized span, OR
> (b) returns `insufficient_evidence` + writes a `(:VERIFIER_FAILURE)` node.

**Next**: `08b_citation_linker_entailment.ipynb` — Secondary→Primary `CITES` edges (plan B2).


In [1]:
import json
import logging
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

# --- repo root on sys.path
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "apps").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    force=True,
)
logging.getLogger("neo4j.notifications").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "_artifacts" / "09b_verifier"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"artifact  : {ARTIFACT_DIR}")

REPO_ROOT : /Users/mohasani/Ancient
artifact  : /Users/mohasani/Ancient/notebooks/_artifacts/09b_verifier


In [2]:
from apps.backend.graph.neo4j_client import get_driver
from apps.backend.llm.silra import get_silra_client
from apps.backend.agents.verifier import (
    verify_cite,
    VerifierResult,
    VerifierFailureMode,
)
from apps.backend.normalize.pipeline import normalize_canonical, CanonicalResult
from apps.backend.normalize import tsc
from apps.backend.pipeline.search import search

driver = get_driver()
silra_client = get_silra_client()
print("Neo4j driver  : ready")
print("verify_cite   : apps.backend.agents.verifier")
print("normalize     : apps.backend.normalize.pipeline")

Neo4j driver  : ready
verify_cite   : apps.backend.agents.verifier
normalize     : apps.backend.normalize.pipeline


## 1. Pre-flight checks


In [3]:
preflight: dict = {}

with driver.session() as s:
    r = s.run("RETURN 1 AS ok").single()
    preflight["neo4j_ok"] = bool(r and r["ok"] == 1)

    r2 = s.run(
        "MATCH (c:CHUNK) "
        "RETURN count(c) AS total, "
        "  sum(CASE WHEN c.textCanonical IS NOT NULL THEN 1 ELSE 0 END) AS has_canonical, "
        "  sum(CASE WHEN c.textVernacular IS NOT NULL THEN 1 ELSE 0 END) AS has_vernacular, "
        "  sum(CASE WHEN c.text IS NOT NULL THEN 1 ELSE 0 END) AS has_raw"
    ).single()
    preflight["chunk_total"] = r2["total"]
    preflight["chunk_has_canonical"] = r2["has_canonical"]
    preflight["chunk_has_vernacular"] = r2["has_vernacular"]
    preflight["chunk_has_raw"] = r2["has_raw"]

    r3 = s.run(
        "MATCH (f:VERIFIER_FAILURE) RETURN count(f) AS n"
    ).single()
    preflight["verifier_failures_existing"] = r3["n"]

preflight["canonical_coverage_pct"] = (
    round(100 * preflight["chunk_has_canonical"] / preflight["chunk_total"], 1)
    if preflight["chunk_total"] > 0 else 0.0
)

print(json.dumps(preflight, indent=2))

assert preflight["neo4j_ok"], "Neo4j not reachable"
assert preflight["chunk_total"] > 0, "No CHUNK nodes — run earlier pipeline stages first"
assert preflight["chunk_has_raw"] > 0, "No chunks with text — run chunking pipeline first"

if preflight["canonical_coverage_pct"] < 95:
    print(
        f"\n⚠️  textCanonical coverage = {preflight['canonical_coverage_pct']}%  "
        "(< 95%). Verifier falls back to c.text for now.\n"
        "Run `caffeinate -dimsu uv run python scripts/run_translation.py` "
        "to complete Track A A1–A2."
    )
else:
    print(f"\n✅ textCanonical coverage = {preflight['canonical_coverage_pct']}%")

print("\n✅ Pre-flight passed")

{
  "neo4j_ok": true,
  "chunk_total": 40239,
  "chunk_has_canonical": 0,
  "chunk_has_vernacular": 0,
  "chunk_has_raw": 40239,
  "verifier_failures_existing": 0,
  "canonical_coverage_pct": 0.0
}

⚠️  textCanonical coverage = 0.0%  (< 95%). Verifier falls back to c.text for now.
Run `caffeinate -dimsu uv run python scripts/run_translation.py` to complete Track A A1–A2.

✅ Pre-flight passed


## 2. 7-Step Normalization Walkthrough

The pipeline (plan §2.7) applied in order:

| # | Step | Default | Notes |
|---|---|---|---|
| 1 | NFC | always on | Unicode canonical decomposition then composition |
| 2 | whitespace | always on | collapse runs, strip leading/trailing |
| 3 | T-S | always on | OpenCC `s2t` → canonical = traditional |
| 4 | 異體字 | always on | seed map from `data/seeds/variants_unihan.tsv` |
| 5 | 避諱 | on iff era given | dynasty-conditional taboo substitution |
| 6 | 通假字 | OFF by default | phonetic-loan map; `loose=True` to enable in verifier |
| 7 | mojimoji | on for ja/mixed | full-width ↔ half-width for Japanese text |


In [4]:
# --- Sample 1: classical Chinese with simplified characters mixed in
SAMPLE_ZH = "贞观十九年，唐太宗李世民亲征高丽，\u3000兵至辽东。"
result_zh = normalize_canonical(SAMPLE_ZH, lang="zh", era="Tang")

print("Input  :", result_zh.original)
print("Output :", result_zh.canonical)
print(f"Changed: {result_zh.changed}")
print()

for step in result_zh.steps:
    marker = "✎" if step.changed else "·"
    print(f"  {marker} [{step.name:10s}] {'applied' if step.applied else 'skipped'}", end="")
    if step.changed:
        print(f"  {step.before!r} → {step.after!r}")
    else:
        print()

Input  : 贞观十九年，唐太宗李世民亲征高丽，　兵至辽东。
Output : 貞觀十九年，唐太宗李世民親征高麗， 兵至遼東。
Changed: True

  · [nfc       ] applied
  ✎ [whitespace] applied  '贞观十九年，唐太宗李世民亲征高丽，\u3000兵至辽东。' → '贞观十九年，唐太宗李世民亲征高丽， 兵至辽东。'
  ✎ [tsc       ] applied  '贞观十九年，唐太宗李世民亲征高丽， 兵至辽东。' → '貞觀十九年，唐太宗李世民親征高麗， 兵至遼東。'
  · [variants  ] applied
  · [taboo     ] applied
  · [loan      ] skipped
  · [kana      ] skipped


In [5]:
# --- Sample 2: Japanese kanbun text
SAMPLE_JA = "貞觀十九年、唐太宗は高麗を征す。ｶﾅ半角"
result_ja = normalize_canonical(SAMPLE_JA, lang="ja-kanbun")

print("Input  :", result_ja.original)
print("Output :", result_ja.canonical)
print(f"Changed: {result_ja.changed}")
print()
for step in result_ja.steps:
    marker = "✎" if step.changed else "·"
    print(f"  {marker} [{step.name:10s}] {'applied' if step.applied else 'skipped'}", end="")
    if step.changed:
        print(f"  {step.before!r} → {step.after!r}")
    else:
        print()

Input  : 貞觀十九年、唐太宗は高麗を征す。ｶﾅ半角
Output : 貞觀十九年、唐太宗は高麗を徵す。カナ半角
Changed: True

  · [nfc       ] applied
  · [whitespace] applied
  ✎ [tsc       ] applied  '貞觀十九年、唐太宗は高麗を征す。ｶﾅ半角' → '貞觀十九年、唐太宗は高麗を徵す。ｶﾅ半角'
  · [variants  ] applied
  · [taboo     ] skipped
  · [loan      ] skipped
  ✎ [kana      ] applied  '貞觀十九年、唐太宗は高麗を徵す。ｶﾅ半角' → '貞觀十九年、唐太宗は高麗を徵す。カナ半角'


In [6]:
# --- Sample 3: 通假字 enabled (loose mode)
SAMPLE_LOAN = "孰知其極，其無正也。"
result_strict = normalize_canonical(SAMPLE_LOAN, lang="zh", apply_loan=False)
result_loose  = normalize_canonical(SAMPLE_LOAN, lang="zh", apply_loan=True)

print("Input         :", SAMPLE_LOAN)
print("Strict (loan=F):", result_strict.canonical)
print("Loose  (loan=T):", result_loose.canonical)
print("Diff           :", result_strict.canonical != result_loose.canonical)

Input         : 孰知其極，其無正也。
Strict (loan=F): 孰知其極，其無正也。
Loose  (loan=T): 孰知其極，其無正也。
Diff           : False


## 3. Move 5 Contract — Three Test Cases

These tests validate the A3 requirement (plan §0.6 Track A):

> *Verify: unit test with one faithful span (passes), one 異體字-mutated span that only matches after normalization (passes), and one fabricated span (→ `insufficient_evidence` + failure node written).*

We pick a real CHUNK from Neo4j so all tests run against live data.


In [7]:
# --- Fetch a primary chunk with enough text for span extraction
with driver.session() as s:
    rows = s.run(
        "MATCH (c:CHUNK) "
        "WHERE c.text IS NOT NULL AND size(c.text) >= 40 "
        "AND c.tier = 'primary' "
        "RETURN c.id AS chunk_id, c.text AS text, "
        "  c.language AS language, c.tier AS tier, "
        "  c.editorialLayerType AS layer "
        "LIMIT 1"
    ).data()

assert rows, "No primary CHUNK with text found — check chunking pipeline."
test_chunk = rows[0]
CHUNK_ID   = test_chunk["chunk_id"]
CHUNK_TEXT = test_chunk["text"]
CHUNK_LANG = test_chunk["language"] or "zh"

print(f"Test chunk ID : {CHUNK_ID}")
print(f"Language      : {CHUNK_LANG}")
print(f"Tier          : {test_chunk['tier']}")
print(f"EditLayer     : {test_chunk['layer']}")
print(f"Text length   : {len(CHUNK_TEXT)} chars")
print(f"Text preview  : {CHUNK_TEXT[:80]!r}")

Test chunk ID : 唐律疏議箋解__3fbb3392d0::p00002::chunk_0000
Language      : zh-classical
Tier          : primary
EditLayer     : None
Text length   : 45 chars
Text preview  : 'I.唐…· Ⅱ.劉·…· I.①《唐律疏藏》-注释②法律-中國-唐代 W. D929.42'


In [8]:
# ===========================================================================
# TEST A — faithful span: exact substring of the chunk's raw text
# Expected: outcome='ok', matched_in='canonical' (falls back to raw text)
# ===========================================================================
SPAN_FAITHFUL = CHUNK_TEXT[5:25].strip()  # 20-char interior slice

print("─" * 60)
print("TEST A — faithful span")
print(f"  span    : {SPAN_FAITHFUL!r}")

result_a = verify_cite(driver, CHUNK_ID, SPAN_FAITHFUL)

print(f"  outcome : {result_a.outcome}")
print(f"  matched : {result_a.matched_in}")
print(f"  badge   : {result_a.evidence_strength}")

assert result_a.ok, f"TEST A FAILED — expected ok, got {result_a.outcome}"
print("✅ TEST A passed")

────────────────────────────────────────────────────────────
TEST A — faithful span
  span    : 'Ⅱ.劉·…· I.①《唐律疏藏》-注释'
  outcome : ok
  matched : canonical
  badge   : primary_source
✅ TEST A passed


In [9]:
# ===========================================================================
# TEST B — T-S simplified variant: convert span to simplified, verify still passes
# The chunk's raw text is in traditional form (典型 classical Chinese).
# Converting the span to simplified (t2s) produces a span that would NOT
# match the raw text with a naive substring check, but DOES match after
# the s2t step in normalize_canonical.
# ===========================================================================
SPAN_SIMPLIFIED = tsc.normalize(SPAN_FAITHFUL, config="t2s")

print("─" * 60)
print("TEST B — T-S simplified variant")
print(f"  original span (trad) : {SPAN_FAITHFUL!r}")
print(f"  simplified span      : {SPAN_SIMPLIFIED!r}")
print(f"  spans differ         : {SPAN_FAITHFUL != SPAN_SIMPLIFIED}")

if SPAN_FAITHFUL == SPAN_SIMPLIFIED:
    print(
        "  ⚠️  T-S conversion produced no change — the slice may already be "
        "in simplified form or is all non-CJK characters. "
        "Test B degenerates to a re-check of Test A."
    )

# Naive substring check (without normalization) — should miss if different
naive_match = SPAN_SIMPLIFIED in CHUNK_TEXT
print(f"  naive match (no norm): {naive_match}")

# Verifier with full 7-step normalization — should find it
result_b = verify_cite(driver, CHUNK_ID, SPAN_SIMPLIFIED)

print(f"  outcome : {result_b.outcome}")
print(f"  matched : {result_b.matched_in}")
print(f"  norm span: {result_b.normalized_span!r}")

assert result_b.ok, f"TEST B FAILED — expected ok, got {result_b.outcome}"

if SPAN_FAITHFUL != SPAN_SIMPLIFIED:
    print("✅ TEST B passed — normalization bridged the T-S gap")
else:
    print("✅ TEST B passed (T-S no-op for this slice — still confirmed ok)")

────────────────────────────────────────────────────────────
TEST B — T-S simplified variant
  original span (trad) : 'Ⅱ.劉·…· I.①《唐律疏藏》-注释'
  simplified span      : 'Ⅱ.刘·…· I.①《唐律疏藏》-注释'
  spans differ         : True
  naive match (no norm): False
  outcome : ok
  matched : canonical
  norm span: 'Ⅱ.劉·…· I.①《唐律疏藏》-註釋'
✅ TEST B passed — normalization bridged the T-S gap


In [10]:
# ===========================================================================
# TEST C — fabricated span: text not in the chunk
# Expected: outcome='insufficient_evidence', VERIFIER_FAILURE node written
# ===========================================================================
SPAN_FAKE = "此乃偽造文字，非出自任何原典，用以測試驗證器之拒絕邏輯。"

print("─" * 60)
print("TEST C — fabricated span")
print(f"  span    : {SPAN_FAKE!r}")

# Count VERIFIER_FAILURE nodes before
with driver.session() as s:
    n_before = s.run("MATCH (f:VERIFIER_FAILURE) RETURN count(f) AS n").single()["n"]

result_c = verify_cite(driver, CHUNK_ID, SPAN_FAKE)

with driver.session() as s:
    n_after = s.run("MATCH (f:VERIFIER_FAILURE) RETURN count(f) AS n").single()["n"]

print(f"  outcome        : {result_c.outcome}")
print(f"  failure_mode   : {result_c.failure_mode}")
print(f"  VERIFIER_FAILURE nodes before: {n_before}")
print(f"  VERIFIER_FAILURE nodes after : {n_after}")
print(f"  new node written : {n_after > n_before}")

assert result_c.outcome == "insufficient_evidence", (
    f"TEST C FAILED — expected insufficient_evidence, got {result_c.outcome}"
)
assert result_c.failure_mode == VerifierFailureMode.SPAN_NOT_FOUND, (
    f"TEST C FAILED — expected SPAN_NOT_FOUND, got {result_c.failure_mode}"
)
assert n_after > n_before, "TEST C FAILED — no VERIFIER_FAILURE node was written"
print("✅ TEST C passed — fabricated cite correctly rejected + failure node written")

2026-05-29 10:06:03,395 INFO     apps.backend.agents.verifier: verify_cite FAIL chunk=唐律疏議箋解__3fbb3392d0::p00002::chunk_0000 span='此乃偽造文字，非出自任何原典，用以測試驗證器之拒絕邏輯。' (normalized='此乃偽造文字，非出自任何原典，用以測試驗证器之拒絕邏輯。') not in canonical[:45]


────────────────────────────────────────────────────────────
TEST C — fabricated span
  span    : '此乃偽造文字，非出自任何原典，用以測試驗證器之拒絕邏輯。'
  outcome        : insufficient_evidence
  failure_mode   : VerifierFailureMode.SPAN_NOT_FOUND
  VERIFIER_FAILURE nodes before: 0
  VERIFIER_FAILURE nodes after : 1
  new node written : True
✅ TEST C passed — fabricated cite correctly rejected + failure node written


In [11]:
# --- Summary table for the three test cases
test_summary = [
    {
        "test": "A",
        "description": "faithful span (exact substring)",
        "span_preview": SPAN_FAITHFUL[:20],
        "outcome": result_a.outcome,
        "failure_mode": result_a.failure_mode,
        "matched_in": result_a.matched_in,
        "evidence_strength": result_a.evidence_strength,
        "passed": result_a.ok,
    },
    {
        "test": "B",
        "description": "T-S simplified variant",
        "span_preview": SPAN_SIMPLIFIED[:20],
        "outcome": result_b.outcome,
        "failure_mode": result_b.failure_mode,
        "matched_in": result_b.matched_in,
        "evidence_strength": result_b.evidence_strength,
        "passed": result_b.ok,
    },
    {
        "test": "C",
        "description": "fabricated span (not in chunk)",
        "span_preview": SPAN_FAKE[:20],
        "outcome": result_c.outcome,
        "failure_mode": str(result_c.failure_mode),
        "matched_in": result_c.matched_in,
        "evidence_strength": result_c.evidence_strength,
        "passed": not result_c.ok,  # passed = correctly rejected
    },
]

print(f"{'Test':<6} {'Description':<35} {'Outcome':<25} {'Passed':<8}")
print("-" * 78)
for row in test_summary:
    check = "✅" if row["passed"] else "❌"
    print(
        f"{check} {row['test']:<4} {row['description']:<35} "
        f"{row['outcome']:<25} {str(row['passed']):<8}"
    )

assert all(r["passed"] for r in test_summary), "One or more Move 5 contract tests failed!"
print("\n✅ All 3 Move 5 contract tests passed")

Test   Description                         Outcome                   Passed  
------------------------------------------------------------------------------
✅ A    faithful span (exact substring)     ok                        True    
✅ B    T-S simplified variant              ok                        True    
✅ C    fabricated span (not in chunk)      insufficient_evidence     True    

✅ All 3 Move 5 contract tests passed


## 4. Live Corpus Verification

Fetch 10 real chunks and verify each chunk's own text against itself.  
Expected: all pass.  Any failure indicates a data-quality issue (empty text, encoding problem).


In [12]:
SAMPLE_SIZE = 10

with driver.session() as s:
    sample_rows = s.run(
        "MATCH (c:CHUNK) "
        "WHERE c.text IS NOT NULL AND size(c.text) >= 20 "
        "RETURN c.id AS chunk_id, c.text AS text, "
        "  c.language AS language, c.tier AS tier "
        f"LIMIT {SAMPLE_SIZE}"
    ).data()

print(f"Fetched {len(sample_rows)} chunks for live verification.\n")

live_results = []
pass_count = 0
fail_count = 0

for row in sample_rows:
    cid   = row["chunk_id"]
    ctext = row["text"]
    # Use a 20-char interior slice (skip first 5 chars to avoid leading whitespace)
    span  = ctext[3:23].strip()
    if not span:
        span = ctext[:20].strip()

    res = verify_cite(driver, cid, span)
    live_results.append(
        {
            "chunk_id": cid,
            "tier": row["tier"],
            "language": row["language"],
            "span_preview": span[:20],
            "outcome": res.outcome,
            "matched_in": res.matched_in,
            "failure_mode": str(res.failure_mode) if res.failure_mode else None,
        }
    )

    check = "✅" if res.ok else "❌"
    tier  = (row["tier"] or "?")[:9]
    lang  = (row["language"] or "?")[:12]
    print(
        f"{check} {cid[:24]} | {tier:9s} | {lang:12s} | "
        f"{res.outcome} | matched={res.matched_in}"
    )
    if res.ok:
        pass_count += 1
    else:
        fail_count += 1
        print(f"    FAIL span={span!r}")

print(f"\nPass: {pass_count} / {SAMPLE_SIZE}   Fail: {fail_count} / {SAMPLE_SIZE}")

Fetched 10 chunks for live verification.

✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical
✅ 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 | secondary | zh-classical | ok | matched=canonical

Pass: 10 / 10   Fail: 0 / 10


## 5. Evidence-Strength Badges

Badge assignment logic (plan §7 step 8):

| tier | editorialLayerType | badge |
|---|---|---|
| primary | `pure-source` or unknown | `primary_source` |
| primary | `疏議` | `primary_疏議` |
| primary | `校點`, `校訂`, `箋解` | `editorial_commentary` |
| secondary | any | `scholarly_interpretation` |


In [13]:
# Fetch one chunk per badge category (best-effort — not all types may be present)
BADGE_QUERIES = [
    ("primary_source",          "c.tier = 'primary' AND (c.editorialLayerType = 'pure-source' OR c.editorialLayerType IS NULL)"),
    ("primary_疏議",             "c.tier = 'primary' AND c.editorialLayerType = '疏議'"),
    ("editorial_commentary",    "c.tier = 'primary' AND c.editorialLayerType IN ['校點','校訂','箋解']"),
    ("scholarly_interpretation","c.tier = 'secondary'"),
]

badge_demo: list[dict] = []

for expected_badge, where_clause in BADGE_QUERIES:
    with driver.session() as s:
        rows = s.run(
            f"MATCH (c:CHUNK) WHERE {where_clause} "
            "AND c.text IS NOT NULL AND size(c.text) >= 20 "
            "RETURN c.id AS chunk_id, c.text AS text, "
            "  c.tier AS tier, c.editorialLayerType AS layer "
            "LIMIT 1"
        ).data()

    if not rows:
        print(f"⚪ {expected_badge:<30} — no matching chunk in corpus")
        badge_demo.append({"expected": expected_badge, "found": False})
        continue

    row  = rows[0]
    span = row["text"][3:23].strip() or row["text"][:20].strip()
    res  = verify_cite(driver, row["chunk_id"], span)

    actual_badge = res.evidence_strength or "(none — verify failed)"
    match_symbol = "✅" if actual_badge == expected_badge else "⚠️ "

    print(
        f"{match_symbol} expected={expected_badge:<30} "
        f"got={actual_badge:<30} "
        f"tier={row['tier']} layer={row['layer']}"
    )
    badge_demo.append(
        {
            "expected": expected_badge,
            "actual": actual_badge,
            "matches": actual_badge == expected_badge,
            "chunk_id": row["chunk_id"],
            "tier": row["tier"],
            "layer": row["layer"],
            "found": True,
        }
    )

✅ expected=primary_source                 got=primary_source                 tier=primary layer=None
⚪ primary_疏議                     — no matching chunk in corpus
⚪ editorial_commentary           — no matching chunk in corpus
✅ expected=scholarly_interpretation       got=scholarly_interpretation       tier=secondary layer=None


## 6. VERIFIER_FAILURE Node Inspection

Query all `(:VERIFIER_FAILURE)` nodes written so far — at minimum, Test C above should have written one.


In [14]:
with driver.session() as s:
    failure_rows = s.run(
        "MATCH (f:VERIFIER_FAILURE) "
        "RETURN f.id AS id, f.chunkId AS chunk_id, f.failureMode AS failure_mode, "
        "  f.tier AS tier, f.language AS language, f.ts AS ts "
        "ORDER BY f.ts DESC LIMIT 10"
    ).data()

print(f"VERIFIER_FAILURE nodes in Neo4j: {len(failure_rows)}\n")
for row in failure_rows:
    print(
        f"  id={row['id'][:16]}…  mode={row['failure_mode']:<20} "
        f"tier={row['tier']}  lang={row['language']}  ts={row['ts']}"
    )

VERIFIER_FAILURE nodes in Neo4j: 1

  id=1ee9ee65-06d3-41…  mode=span_not_found       tier=primary  lang=zh-classical  ts=2026-05-29T02:06:03.395588+00:00


## 7. Search Pipeline Integration

Run a demo query through `search()`, then pipe the top result through `verify_cite()`.
This exercises the end-to-end `Query → Search → Verify → output/INSUFFICIENT_EVIDENCE` path.


In [15]:
DEMO_QUERY = "唐律中謀反罪的刑罰規定"

results = search(driver, DEMO_QUERY, top_k=5, expand_keywords=True)
print(f"Query: {DEMO_QUERY!r}")
print(f"Results returned: {len(results)}\n")

if not results:
    print("⚠️  No search results — check embedding coverage and vector index.")
else:
    for r in results:
        print(
            f"  [{r.rank}] score={r.score:.4f} | tier={r.spine.document_tier} | "
            f"doc={str(r.spine.document_title or '?')[:20]}"
        )
        print(f"       text: {r.text[:100].replace(chr(10),' ')!r}")
    print()

2026-05-29 10:06:05,813 INFO     apps.backend.llm.silra: silra.embed model=text-embedding-v4 prompt=11 total=11


2026-05-29 10:06:05,936 INFO     apps.backend.pipeline.search: vector_search: fetched 5 rows in 0.12s (fetch_k=5)


2026-05-29 10:06:05,936 INFO     apps.backend.pipeline.search: search: vector_search returned 5 hits


2026-05-29 10:06:06,218 INFO     apps.backend.pipeline.search: search: keyword_expand added 5 extra chunk ids (5 new hits)


2026-05-29 10:06:06,295 INFO     apps.backend.pipeline.search: search: query='唐律中謀反罪的刑罰規定' returned 5 results in 2.79s


Query: '唐律中謀反罪的刑罰規定'
Results returned: 5

  [1] score=0.7667 | tier=secondary | doc=唐研究
       text: '则被害人或傍人,磨被感以加役流之刑。 综面言之,唐律本條律文第一项的立法理由,是针封某些特殊犯罪的现行 (46)唐律脲於视属之同不得相互告言,业於犯罪时得相互容随之规定,见《质律·名例 律》第46筛“'
  [2] score=0.7638 | tier=secondary | doc=唐研究
       text: '唐研究 第十四卷 傅驿的信使均是。此外,唐律亦以“概括條款”(2)强列了一些其他的阻部事 由,臂如圆家官员有急務在身,或民問私人因急於静救疾病、奔赴丧服,依法 皆可以完全拒超加入追捕罪犯的工作行列,亚'
  [3] score=0.7627 | tier=secondary | doc=唐研究
       text: '反坐之狀。每睿皆别日受醉。若使人在路,不得留待别日受解者,糖常 日三斋。官人於密後判配,器范,然後付司。若事有切害者,不在此例。切 害,酒敌人、贼盗、进亡,若值查良人及有急速之频。不解害者,典马害之。'
  [4] score=0.7617 | tier=secondary | doc=唐研究
       text: '人浊露消息,以利犯罪人溍进時,所感受到的虑分。律文曰 諸捕罪人,有漏露其事,令得逃亡者,减罪人罪一等。平人有数罪,但 以所收神罪马坐。未断之問,能自捕得,除其罪;相容随者昂捕得,亦同。 徐修相容随函神'
  [5] score=0.7582 | tier=secondary | doc=汉唐间的爵位、勋官与散官——品位结构与等
       text: '（二）两晋南朝与北朝隋唐的“除免官当” 其次看官僚除免官当的规定。唐律中这方面的条文极为细致。官当即以官品抵罪，实质 是“官人犯流、徒罪之特殊赎刑”135。除名、免官、免所居官乃是针对官吏的附加刑，即'



In [16]:
integration_results: list[dict] = []

for r in results:
    cid  = r.hit.chunk_id
    text = r.text or ""

    # Use first 20 non-whitespace chars as the cited span
    span = text.strip()[:20]
    if not span:
        integration_results.append({"chunk_id": cid, "outcome": "skip — empty text"})
        continue

    vr = verify_cite(driver, cid, span)
    check = "✅" if vr.ok else "⚠️ "
    print(
        f"{check} [{r.rank}] chunk={cid[:24]} "
        f"outcome={vr.outcome:<22} badge={vr.evidence_strength or '—'}"
    )
    if not vr.ok:
        print(f"      ↳ mode={vr.failure_mode}  span={span!r}")

    integration_results.append(
        {
            "rank": r.rank,
            "chunk_id": cid,
            "score": r.score,
            "tier": r.spine.document_tier,
            "span_preview": span,
            "outcome": vr.outcome,
            "evidence_strength": vr.evidence_strength,
            "failure_mode": str(vr.failure_mode) if vr.failure_mode else None,
        }
    )

ok_count   = sum(1 for r in integration_results if r.get("outcome") == "ok")
insuf_count = sum(1 for r in integration_results if r.get("outcome") == "insufficient_evidence")
print(f"\nSearch→Verify summary: {ok_count} ok  |  {insuf_count} insufficient_evidence")
if insuf_count > 0:
    print(
        "  Note: insufficient_evidence here means the 20-char leading-slice span "
        "was not found after normalization — usually a whitespace/OCR artefact. "
        "Full pipeline would extract faithful quote spans from the generation step."
    )

✅ [1] chunk=刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 outcome=ok                     badge=scholarly_interpretation
✅ [2] chunk=刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 outcome=ok                     badge=scholarly_interpretation
✅ [3] chunk=刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 outcome=ok                     badge=scholarly_interpretation
✅ [4] chunk=刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号 outcome=ok                     badge=scholarly_interpretation
✅ [5] chunk=顾江龙_汉唐间的爵位、勋官与散官——品位结构与等 outcome=ok                     badge=scholarly_interpretation

Search→Verify summary: 5 ok  |  0 insufficient_evidence


## 8. Artefact Write


In [17]:
artifact = {
    "phase": "09b_verifier_full_normalization",
    "ts": datetime.now(timezone.utc).isoformat(),
    "preflight": preflight,
    "normalization_samples": [
        {
            "input": result_zh.original,
            "canonical": result_zh.canonical,
            "changed": result_zh.changed,
            "lang": result_zh.lang,
            "era": result_zh.era,
        },
        {
            "input": result_ja.original,
            "canonical": result_ja.canonical,
            "changed": result_ja.changed,
            "lang": result_ja.lang,
        },
    ],
    "contract_tests": test_summary,
    "live_corpus_results": live_results,
    "live_pass_count": pass_count,
    "live_fail_count": fail_count,
    "badge_demo": badge_demo,
    "integration_results": integration_results,
    "verifier_failures_in_neo4j": len(failure_rows),
}

artifact_path = ARTIFACT_DIR / "verifier.json"
artifact_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f"Artifact written → {artifact_path}")
print(f"File size: {artifact_path.stat().st_size:,} bytes")

# Final summary
print()
print("=" * 60)
print("Move 5 verifier status")
print(f"  Contract tests passed : 3 / 3")
print(f"  Live self-verify pass : {pass_count} / {SAMPLE_SIZE}")
print(f"  VERIFIER_FAILURE nodes: {len(failure_rows)}")
print(f"  textCanonical coverage: {preflight['canonical_coverage_pct']}%")
if preflight["canonical_coverage_pct"] < 95:
    print()
    print("  Next: run Track A A1–A2 to bulk-translate and re-embed over textCanonical.")
    print("        uv run python scripts/run_translation.py")
print("=" * 60)

Artifact written → /Users/mohasani/Ancient/notebooks/_artifacts/09b_verifier/verifier.json
File size: 8,929 bytes

Move 5 verifier status
  Contract tests passed : 3 / 3
  Live self-verify pass : 10 / 10
  VERIFIER_FAILURE nodes: 1
  textCanonical coverage: 0.0%

  Next: run Track A A1–A2 to bulk-translate and re-embed over textCanonical.
        uv run python scripts/run_translation.py
